In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
tokenizer.decode([128007])

'<|end_header_id|>'

In [4]:
import json

with open("/home/wangxi/project/LLM/ChemGFN/data/sft_dataset_chirality_test_sampled.json") as f:
    data = json.load(f)

In [5]:
smi_list = [x["output"] for x in data]

In [7]:
token_ids_list = []
for smi in smi_list:
    smis = [tokenizer.decode(x) for x in tokenizer.encode(smi)]
    if "125" in smis:
        print(smi)
        print(smis)
        break
    # token_ids_list.extend([tokenizer.decode(x) for x in tokenizer.encode(smi)])

C1=CC(=C(C=C1C[C@@H](C(=O)N[C@@H](CC(=O)O)C(=O)O)NC(=O)[C@H]([C@@H](CCS(=O)(=O)[O-])[NH3+])S)[125I])O
['<|begin_of_text|>', 'C', '1', '=', 'CC', '(=', 'C', '(C', '=C', '1', 'C', '[C', '@@', 'H', '](', 'C', '(=', 'O', ')', 'N', '[C', '@@', 'H', '](', 'CC', '(=', 'O', ')', 'O', ')', 'C', '(=', 'O', ')', 'O', ')', 'NC', '(=', 'O', ')[', 'C', '@', 'H', ']', '([', 'C', '@@', 'H', '](', 'CC', 'S', '(=', 'O', ')(', '=', 'O', ')[', 'O', '-', '])[', 'NH', '3', '+', '])', 'S', ')[', '125', 'I', '])', 'O']


In [11]:
list((set(token_ids_list)))

['CCI',
 'OC',
 'CO',
 '75',
 'Ni',
 '8',
 'c',
 '(Cl',
 'B',
 '[G',
 '52',
 'Co',
 '125',
 '/O',
 'i',
 'Sn',
 'As',
 '43',
 'SCP',
 'PF',
 '4',
 '131',
 '(S',
 'NF',
 '([',
 '@@',
 'u',
 '66',
 'OSC',
 'PH',
 'W',
 '91',
 '92',
 '51',
 '45',
 '96',
 '22',
 '\\S',
 '/S',
 'ONO',
 '[S',
 'NSS',
 'S',
 ')/',
 'CNN',
 '(B',
 'Cr',
 '(OS',
 '71',
 '(F',
 '/C',
 'OI',
 'PO',
 'f',
 '14',
 'SN',
 'F',
 '53',
 ')\\',
 'OO',
 'NOP',
 'Ir',
 '31',
 '93',
 '(C',
 '35',
 '[C',
 'ICI',
 '<|begin_of_text|>',
 'Sm',
 'La',
 '+]',
 '[H',
 'SO',
 'NH',
 '48',
 '11',
 '84',
 'OSP',
 'CSI',
 'Ti',
 'Re',
 'T',
 'Y',
 '13',
 '][',
 '68',
 '123',
 '[N',
 '](',
 'Lu',
 'In',
 '(=',
 '+',
 '(/',
 'Th',
 'Sc',
 'CSS',
 'ONS',
 '/N',
 '[I',
 '.P',
 'PPP',
 '=C',
 '.I',
 'NI',
 '[',
 'R',
 '63',
 'OB',
 '#[',
 '/',
 'SCO',
 'H',
 '225',
 '7',
 'Nd',
 'Os',
 '\\',
 '/I',
 '(',
 '])/',
 'P',
 '.Br',
 '=N',
 'SI',
 'NO',
 '64',
 'IN',
 '34',
 ']/',
 '#',
 '.CON',
 '85',
 'Br',
 '6',
 '124',
 'Pt',
 'N',
 'Al',
 

In [22]:
with open("./assets/llama3-smiles-vocab.txt", "w") as f:
    for token_id in list((set(token_ids_list))):
        f.write(token_id + "\n")

## model test

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
[tokenizer.decode(x) for x in tokenizer.encode("CC1=C(NCCS1)C(C)C")]

['<|begin_of_text|>',
 'CC',
 '1',
 '=C',
 '(N',
 'CC',
 'S',
 '1',
 ')',
 'C',
 '(C',
 ')',
 'C']

In [3]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "'You must follow the following rules to generate SMILES:1. **Basic Structure:**   - SMILES is a line notation using printable characters without spaces.   - Represents molecules and reactions.2. **Atoms Representation:**   - Atoms are represented by their atomic symbols.   - Non-hydrogen atoms are enclosed in square brackets, e.g., [C], [O].   - Elements in the organic subset (B, C, N, O, P, S, F, Cl, Br, I) can be written without brackets if they conform to normal valences.3. **Hydrogens and Charges:**   - Attached hydrogens are shown by H followed by a digit (optional).   - Formal charges are shown by + or -, followed by a digit (optional).   - Example: [Fe+++] is the same as [Fe+3].4. **Bonds Representation:**   - Single: -   - Double: =   - Triple: #   - Aromatic: :   - Adjacent atoms are assumed to be connected by a single or aromatic bond if no bond symbol is present.5. **Branches and Cyclic Structures:**   - Branches are enclosed in parentheses and can be nested.   - Cyclic structures are represented by breaking one bond in each ring and using digits to indicate ring closure.6. **Disconnected Compounds:**   - Written as individual structures separated by a period (.)7. **Isomer and Chirality Specifications:**   - Chirality is indicated by @ or @@ following the atomic symbol.   - @ indicates anticlockwise; @@ indicates clockwise.   - Absence of chirality specification means chirality is not specified.8. **Isotopic Specifications:**   - Indicated by preceding the atomic symbol with the atomic mass number inside brackets.9. **Double Bond Configuration:**   - Directional bonds are shown by / and \\ to indicate relative directionality.10. **General Rules:**    - Any valid order of SMILES notation is acceptable.    - Implicit hydrogens are assumed unless explicitly stated.    - Matching pairs of digits indicate bonded atoms, and adjacent atoms separated by a period (.) are not bonded.By following these simplified rules, you can effectively use SMILES notation to represent molecular structures.Convert the IUPAC name acetylene;2,5-dimethylhexane;ethene into a SMILES string.There are several examples to convert IUPAC names into SMILES notation.                    The IUPAC name 6-methyl-5-propan-2-yl-3,4-dihydro-2H-1,4-thiazine have its SMILES is CC1=C(NCCS1)C(C)C                     The IUPAC chemical name 4-tert-butyl-6-pyrrolidin-3-ylmorpholin-3-one into its SMILES form is CC(C)(C)N1CC(OCC1=O)C2CCNC2                     The SMILES version of the IUPAC name 2-(4-propan-2-ylphenyl)-1,3-thiazole is CC(C)C1=CC=C(C=C1)C2=NC=CS2You must give only one SMILES string as output without any additional information, explanation, context, and characters.'"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [4]:
output = model(input_ids=input_ids)

In [ ]:
tokenizer.decode(torch.argmax(output.logits, dim=-1)[0])

In [ ]:
outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
response = outputs[0][input_ids.shape[-1]:]

In [ ]:
print(tokenizer.decode(response, skip_special_tokens=False))